- Ficher source : part-des-menages-disposant-au-moins-dune-voiture-taux-de-motorisation-commune.csv
- Fichier de sorti : stg_gouv_voitures.csv
- Date de création : 13/11/2025
- Dernière modification : 17/11/2025

- Version(s) : 
    - 1 - 13/11/2025 : Nettoyage du fichier et création du csv "insee_voiture"
    - 2 - 14/11/2025 : Renommage du fichier csv de sorti en "gouv_voitures"
    - 3 - 17/11/2025 : Renommage du fichier et création du csv "stg_gouv_voitures"

In [1]:
#Librairie(s) utilisée(s)
import pandas as pd

In [2]:
#Création du dataframe à partir du fichier csv
df = pd.read_csv(r'C:\Users\justi\OneDrive\Je-deviens-Data-Analyst\JEDHA\00_Certif\bloc_6\01_data\01_data_bronze\part-des-menages-disposant-au-moins-dune-voiture-taux-de-motorisation-commune.csv', delimiter=',')
df.head()

,date_mesure,geocode_commune,libelle_commune,valeur
0,2016-01-01T00:00:00.000,75056,Paris,0.351572
1,2011-01-01T00:00:00.000,78522,Rochefort-en-Yvelines,0.958449
2,2016-01-01T00:00:00.000,63445,Vassel,0.930435
3,2016-01-01T00:00:00.000,57638,Schœneck,0.938136
4,2016-01-01T00:00:00.000,22317,Saint-Méloir-des-Bois,0.953271


In [3]:
df.shape

(104533, 4)

In [4]:
df.count()

date_mesure        104533
geocode_commune    104533
libelle_commune    104533
valeur             104533
dtype: int64

In [5]:
df.count().isna() #Aucune colonnes null

date_mesure        False
geocode_commune    False
libelle_commune    False
valeur             False
dtype: bool

In [6]:
#Création de la colonne 'MVP' pour cibler les communes cibles et filtrer le dataset dessus
MVP = [35032, 35058, 35196, 35065, 35275, 35144, 35266, 35315, 35278, 35245, 35208, 35353, 35120, 35210, 35352, 35059, 35250, 35079, 35363, 35051, 35055, 35076, 35189, 35066, 35204, 35022, 35351, 35088, 35080, 35131, 35001, 35206, 35081, 35139, 35334, 35216, 35024, 35240, 35039, 35047, 35281, 35180]

#Création du mask pour filtrer sur le département de l'Ille-et-Vilaine
mask = df['geocode_commune'].str[:2] == '35'
df = df[mask]

df['MVP'] = df['geocode_commune'].astype(int).apply(lambda x: 'oui' if x in MVP else 'non') #conversion du type de la colonne 'geocode_commune' en type int pour la comparer avec la variable 'MVP'

df['MVP'].describe()

count     996
unique      2
top       non
freq      870
Name: MVP, dtype: object

In [7]:
#Création du mask pour filtrer sur le département de l'Ille-et-Vilaine
mask = (df['MVP'] == 'oui') & (df['date_mesure'].str[:4] == '2022')
df = df[mask]

df.shape

(42, 5)

In [8]:
#Identification des colonnes à garder
keep_columns = ["geocode_commune","valeur"]
df = df.loc[:,keep_columns]

#Renommage de la colonne INSEE
df.rename(columns={"geocode_commune": "code_geo", "valeur": "taux_motorisation"}, inplace=True)

df.head()

,code_geo,taux_motorisation
165,35275,0.943874
180,35196,0.898676
1941,35204,0.963847
5665,35206,0.914370
6781,35131,0.905405


In [9]:
#Formatage des types de colonnes
for col in df.columns:
    if col == 'code_geo':
        df[col] = df[col].astype(int)
    elif col == 'taux_motorisation':
        df[col] = df[col].astype(float)

df.dtypes

code_geo               int32
taux_motorisation    float64
dtype: object

In [ ]:
#Exporte le dataset nettoyé en csv
df.to_csv("stg_gouv_voitures.csv", sep=";", index=False, encoding='UTF-8')